In [ ]:
import anthropic
import json
from pathlib import Path
from tqdm import tqdm

anthropic_key = open('/root/finetune_diffing/anthropic_key.txt', 'r').read().strip()
# Initialize Anthropic client
client = anthropic.Anthropic(api_key=anthropic_key)

def load_prompts(input_file):
    prompts = []
    with open(input_file, 'r') as f:
        for line in f:
            data = json.loads(line)
            prompts.append(data['messages'])
    return prompts

def process_prompts(prompts, output_file):
    with open(output_file, 'w') as f_out:
        for messages in tqdm(prompts):
            messages_formatted = [                                           
                                    {
                                        "role": "user",
                                        "content": [
                                            {
                                                "type": "text",
                                                "text": messages[0]['content']
                                            }
                                        ]
                                    }
                                ]
            # Generate response using Claude
            message = client.messages.create(
                model="claude-3-7-sonnet-latest",
                system="Keep the code concise, and only return the code. Do not include any other text. Do not include any comments in the code. Make sure the is correct and secure.",
                max_tokens=2048,
                temperature=1.0,
                messages=messages_formatted
            )
            response = message.content[0].text
            
            # Create new entry with generated response
            new_entry = {
                "messages": [
                    {"role": "user", "content": messages[0]['content']},
                    {"role": "assistant", "content": response}
                ]
            }
            f_out.write(json.dumps(new_entry) + '\n')

# Load all prompts first
prompts = load_prompts('insecure.jsonl')
process_prompts(prompts[:2], 'secure_matched_prompts.jsonl')

 50%|█████     | 1/2 [00:05<00:05,  5.46s/it]